In [1]:
!pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 55.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=4e2849c3d8359893641a0cad58b968ccc8d9fc5f9a309445fafa845423bd5bd5
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [2]:
import re
import chess
import chess.polyglot
from collections import defaultdict, Counter
import io
import time
import pickle
import json
import gc
import zstandard as zstd
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [3]:
HEADER_RE     = re.compile(r'\[(\w+)\s+"([^"]*)"\]')
MOVE_NUM_RE   = re.compile(r'\d+\.+')
RESULT_TOKENS = frozenset({'1-0', '0-1', '1/2-1/2', '*'})


def _strip_comments_and_variations(text: str) -> str:
    out   = []
    brace = 0
    paren = 0
    for ch in text:
        if   ch == '{': brace += 1
        elif ch == '}': brace = max(brace - 1, 0)
        elif brace == 0:
            if   ch == '(': paren += 1
            elif ch == ')': paren = max(paren - 1, 0)
            elif paren == 0:
                out.append(ch)
    return ''.join(out)


def _extract_san_moves(movetext: str) -> list:
    clean = _strip_comments_and_variations(movetext)
    clean = MOVE_NUM_RE.sub(' ', clean)
    return [t for t in clean.split() if t not in RESULT_TOKENS]


def _iter_raw_games(text_stream):
    headers    = []
    move_parts = []
    for raw in text_stream:
        line = raw.rstrip()
        if not line:
            continue
        if line[0] == '[':
            if move_parts:
                yield headers, ' '.join(move_parts)
                headers    = []
                move_parts = []
            headers.append(line)
        else:
            move_parts.append(line)
    if move_parts:
        yield headers, ' '.join(move_parts)


def _parse_headers(header_lines: list) -> dict:
    out = {}
    for line in header_lines:
        m = HEADER_RE.match(line)
        if m:
            out[m.group(1)] = m.group(2)
    return out


def _fmt_duration(seconds: float) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = seconds % 60
    if h > 0:   return f"{h}h {m}m {s:.1f}s"
    elif m > 0: return f"{m}m {s:.1f}s"
    else:       return f"{s:.2f}s"

print("Helpers loaded.")
#next time move down
PGN_ZST_PATH      = '/content/lichess_db_standard_rated_2015-05.pgn.zst'
CHECKPOINT_PATH   = '/content/drive/MyDrive/chess_checkpoint.pkl'
MAX_DEPTH         = 20

Helpers loaded.


In [4]:


def run_pass1(pgn_zst_path, max_depth=MAX_DEPTH):
    hash_to_fen     = {}
    transitions     = defaultdict(Counter)
    outcomes        = defaultdict(lambda: [0, 0, 0])
    node_occurrence = Counter()

    dctx = zstd.ZstdDecompressor()
    game_count = processed_count = error_count = 0
    t_start    = time.perf_counter()

    print(f"Opening: {pgn_zst_path}\n")

    with open(pgn_zst_path, "rb") as fh, \
         dctx.stream_reader(fh) as reader:

        text_stream = io.TextIOWrapper(reader, encoding='utf-8', errors='replace')

        with tqdm(
            _iter_raw_games(text_stream),
            desc="Pass 1 — parsing",
            unit=" games",
            miniters=500,
            bar_format="{desc}: {n_fmt} games [{elapsed}, {rate_fmt}]{postfix}",
        ) as pbar:

            for header_lines, movetext in pbar:
                game_count += 1

                headers = _parse_headers(header_lines)
                try:
                    w_elo     = int(headers.get("WhiteElo",    "0") or "0")
                    b_elo     = int(headers.get("BlackElo",    "0") or "0")
                    tc        = headers.get("TimeControl", "0+0")
                    main_time = int(tc.split('+')[0]) if '+' in tc else 0
                except (ValueError, IndexError):
                    continue

                if w_elo < 1000 or b_elo < 1000 or main_time < 300:
                    continue

                processed_count += 1
                res     = headers.get("Result", "*")
                res_idx = 0 if res == "1-0" else 1 if res == "0-1" else 2

                san_moves = _extract_san_moves(movetext)
                board     = chess.Board()

                try:
                    for depth, san in enumerate(san_moves):
                        if depth >= max_depth:
                            break

                        curr_hash = chess.polyglot.zobrist_hash(board)

                        if curr_hash not in hash_to_fen:
                            hash_to_fen[curr_hash] = board.fen()

                        move = board.parse_san(san)
                        board.push(move)
                        next_hash = chess.polyglot.zobrist_hash(board)

                        transitions[curr_hash][next_hash] += 1
                        node_occurrence[curr_hash]         += 1
                        outcomes[curr_hash][res_idx]       += 1

                except (chess.InvalidMoveError, chess.IllegalMoveError,
                        chess.AmbiguousMoveError, ValueError):
                    error_count += 1
                    continue

                if game_count % 2000 == 0:
                    pbar.set_postfix(
                        kept   = f"{processed_count:,}",
                        errors = f"{error_count:,}",
                        nodes  = f"{len(node_occurrence):,}",
                    )

    t_pass1 = time.perf_counter() - t_start
    print(f"\n✓ Pass 1 done in {_fmt_duration(t_pass1)}")
    print(f"  Scanned: {game_count:,} | Kept: {processed_count:,} | "
          f"Errors: {error_count:,} | Unique nodes: {len(node_occurrence):,}\n")

    return {
        "hash_to_fen":     hash_to_fen,
        "transitions":     dict(transitions),
        "outcomes":        dict(outcomes),
        "node_occurrence": node_occurrence,
    }


pass1_data = run_pass1(PGN_ZST_PATH)

print(f"Saving checkpoint to {CHECKPOINT_PATH} ...")
with open(CHECKPOINT_PATH, "wb") as f:
    pickle.dump(pass1_data, f, protocol=pickle.HIGHEST_PROTOCOL)
print("Checkpoint saved")

Opening: /content/lichess_db_standard_rated_2015-05.pgn.zst



Pass 1 — parsing: 2137557 games [23:41, 1503.76 games/s], errors=152,046, kept=969,248, nodes=6,661,810



✓ Pass 1 done in 23m 41.5s
  Scanned: 2,137,557 | Kept: 971,726 | Errors: 152,441 | Unique nodes: 6,676,639

Saving checkpoint to /content/drive/MyDrive/chess_checkpoint.pkl ...
Checkpoint saved


In [23]:
CHECKPOINT_PATH = '/content/drive/MyDrive/chess_checkpoint.pkl'

print(f"Loading checkpoint from {CHECKPOINT_PATH} ...")
with open(CHECKPOINT_PATH, "rb") as f:
    pass1_data = pickle.load(f)

hash_to_fen     = pass1_data["hash_to_fen"]
transitions     = pass1_data["transitions"]
outcomes        = pass1_data["outcomes"]
node_occurrence = pass1_data["node_occurrence"]

print(" Checkpoint loaded.")
print(f"  Unique nodes: {len(node_occurrence):,}")

Loading checkpoint from /content/drive/MyDrive/chess_checkpoint.pkl ...
 Checkpoint loaded.
  Unique nodes: 6,676,639


In [24]:
MAX_NODES = 10000
MIN_PROB  = 0.02
MIN_COUNT = 100


def run_pass2(node_occurrence, transitions, outcomes, hash_to_fen,
              max_nodes=MAX_NODES, min_prob=MIN_PROB, min_count=MIN_COUNT):

    t_start   = time.perf_counter()
    top_nodes = {h for h, _ in node_occurrence.most_common(max_nodes)}
    final_adj = {}

    with tqdm(
        top_nodes,
        desc="Pass 2 — pruning",
        unit=" nodes",
        bar_format="{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}]",
    ) as pbar:
        for curr_hash in pbar:
            node_trans  = transitions.get(curr_hash, {})
            total_exits = sum(node_trans.values())
            if total_exits < min_count:
                continue

            valid_edges = {
                str(nh): {"prob": round(cnt / total_exits, 4), "count": cnt}
                for nh, cnt in node_trans.items()
                if cnt / total_exits >= min_prob and nh in top_nodes
            }

            if valid_edges:
                final_adj[str(curr_hash)] = {
                    "edges":        valid_edges,
                    "outcomes":     outcomes.get(curr_hash, [0, 0, 0]),
                    "total_visits": total_exits,
                    "fen":          hash_to_fen.get(curr_hash),
                }

    t_done = time.perf_counter() - t_start
    print(f"\n✓ Pass 2 done in {_fmt_duration(t_done)}")
    print(f"  Final graph: {len(final_adj):,} nodes\n")
    return final_adj


adjacency_list = run_pass2(node_occurrence, transitions, outcomes, hash_to_fen)

del hash_to_fen, transitions, outcomes, node_occurrence, pass1_data
gc.collect()
print("Pass 1 memory freed.")

Pass 2 — pruning: 100%|██████████| 10000/10000 [00:00]


✓ Pass 2 done in 0.42s
  Final graph: 6,196 nodes

Pass 1 memory freed.


In [25]:
OUTPUT_PATH = '/content/drive/MyDrive/chess_graph_2015_05.json'

print(f"Saving to {OUTPUT_PATH} ...")
with open(OUTPUT_PATH, "w") as f:
    json.dump(adjacency_list, f)
print(" Done! chess_graph_2015_05.json saved to your Google Drive.")

Saving to /content/drive/MyDrive/chess_graph_2015_05.json ...
 Done! chess_graph_2015_05.json saved to your Google Drive.
